# ADWIN — Adaptive Windowing for Concept Drift Detection

> Bifet, A. & Gavalda, R. (2007) *"Learning from Time-Changing Data with Adaptive Windowing"*

---

## Core idea

ADWIN maintains a **variable-length window W** of the most recent observations.  
At each time step it checks every possible split of W into two consecutive sub-windows $W_0$ (older) and $W_1$ (newer) and removes $W_0$ when their means differ significantly.

The decision criterion uses the **Hoeffding / Bernstein bound**:

$$\epsilon_{cut} = \sqrt{\frac{1}{2m^*}\ln\frac{4n}{\delta}}$$

where:
- $m^* = \dfrac{n_0 \cdot n_1}{n_0 + n_1}$ (harmonic-style weight)
- $n = n_0 + n_1 = |W|$
- $\delta$ — confidence (e.g. 0.002)

**Drift detected** when $|\hat{\mu}_{W_0} - \hat{\mu}_{W_1}| \geq \epsilon_{cut}$

---

## ADWIN2 — Bucket-based compression

Storing the raw window costs O(n) memory.  
ADWIN2 compresses the window into **exponential histogram buckets**:

- Bucket at level $k$ holds the sum and count of $2^k$ elements
- At most $M$ buckets per level (M controls accuracy/memory trade-off)
- When a level overflows, two buckets merge into the next level

This gives **O(log n / M)** memory while guaranteeing the same statistical bound.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Imports OK')

---
## 1  ADWIN — Full Implementation

In [ ]:
class ADWIN:
    """
    ADWIN drift detector (Bifet & Gavalda, 2007).

    Parameters
    ----------
    delta : float
        Confidence parameter.  Smaller → fewer false alarms, higher detection lag.
    min_window : int
        Minimum window size before testing cuts.
    """

    def __init__(self, delta: float = 0.002, min_window: int = 10):
        self.delta = delta
        self.min_window = min_window
        self.window: list[float] = []
        self.drift_detected = False
        self.n_detections = 0

    # --- public API ---

    def add_element(self, value: float) -> bool:
        """Add one element and return True if drift was detected."""
        self.window.append(float(value))
        self.drift_detected = False

        if len(self.window) < self.min_window:
            return False

        drift = self._test_and_shrink()
        if drift:
            self.drift_detected = True
            self.n_detections += 1
        return drift

    @property
    def mean(self) -> float:
        return float(np.mean(self.window)) if self.window else 0.0

    @property
    def variance(self) -> float:
        return float(np.var(self.window)) if len(self.window) > 1 else 0.0

    # --- internal ---

    def _test_and_shrink(self) -> bool:
        """Test all cuts; if drift found, drop oldest sub-window."""
        w = np.array(self.window)
        n = len(w)

        for cut in range(1, n):
            w0, w1 = w[:cut], w[cut:]
            n0, n1 = len(w0), len(w1)
            m_star = (n0 * n1) / (n0 + n1)
            eps_cut = np.sqrt(np.log(4.0 * n / self.delta) / (2.0 * m_star))

            if abs(w0.mean() - w1.mean()) >= eps_cut:
                self.window = list(w1)  # drop older sub-window
                return True

        return False


print('ADWIN class ready.')

---
## 2  Abrupt Drift Detection

In [ ]:
def run_adwin(stream, delta=0.002):
    adwin = ADWIN(delta=delta)
    drifts, means = [], []
    for t, val in enumerate(stream):
        adwin.add_element(val)
        if adwin.drift_detected:
            drifts.append(t)
        means.append(adwin.mean)
    return drifts, means


# Abrupt drift at t=300 and t=700
T = 1000
stream_abrupt = np.concatenate([
    np.random.normal(0, 1, 300),
    np.random.normal(4, 1, 400),
    np.random.normal(-2, 1, 300),
])

drifts_abrupt, means_abrupt = run_adwin(stream_abrupt, delta=0.002)
print(f'Abrupt drifts detected at: {drifts_abrupt[:6]}  (true drifts: 300, 700)')

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(stream_abrupt, alpha=0.5, lw=0.7, color='steelblue')
for dp in drifts_abrupt:
    axes[0].axvline(dp, color='red', alpha=0.3, lw=0.7)
axes[0].axvline(300, color='black', ls='--', lw=2, label='True drift')
axes[0].axvline(700, color='black', ls='--', lw=2)
axes[0].set_title('Stream values')
axes[0].legend()

axes[1].plot(means_abrupt, lw=2, color='darkorange', label='ADWIN mean')
axes[1].set_title('ADWIN adaptive window mean')
axes[1].set_xlabel('Time step')
axes[1].legend()

plt.suptitle('ADWIN — Abrupt Drift Detection', fontsize=13)
plt.tight_layout()
plt.show()

---
## 3  Gradual Drift Detection

In [ ]:
# Gradual drift: mean linearly shifts from 0 to 5 between t=200 and t=600
t_axis = np.arange(1000)
mean_signal = np.where(t_axis < 200, 0,
              np.where(t_axis < 600, (t_axis - 200) / 400 * 5, 5))
stream_gradual = mean_signal + np.random.normal(0, 1, 1000)

drifts_grad, means_grad = run_adwin(stream_gradual, delta=0.002)
print(f'Gradual drifts detected: {len(drifts_grad)} events starting at t={drifts_grad[0] if drifts_grad else "—"}')

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(stream_gradual, alpha=0.4, lw=0.7, color='steelblue')
axes[0].plot(mean_signal, lw=2, color='black', label='True mean')
for dp in drifts_grad:
    axes[0].axvline(dp, color='red', alpha=0.2, lw=0.6)
axes[0].set_title('Gradual drift stream')
axes[0].legend()

axes[1].plot(means_grad, lw=2, color='darkorange', label='ADWIN mean')
axes[1].plot(mean_signal, lw=1.5, ls='--', color='black', label='True mean')
axes[1].set_title('ADWIN tracking of gradual drift')
axes[1].set_xlabel('Time step')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 4  Effect of δ on Detection Lag vs False Alarm Rate

In [ ]:
stream_test = np.concatenate([
    np.random.normal(0, 1, 500),
    np.random.normal(3, 1, 500),
])
true_drift = 500

rows = []
for delta in [0.5, 0.1, 0.01, 0.002, 0.0001]:
    drifts, _ = run_adwin(stream_test, delta=delta)
    first_post = [d for d in drifts if d >= true_drift]
    pre_drifts = [d for d in drifts if d < true_drift]
    lag = (first_post[0] - true_drift) if first_post else None
    rows.append({'delta': delta, 'false_alarms': len(pre_drifts), 'detection_lag': lag})

import pandas as pd
df = pd.DataFrame(rows)
print(df.to_string(index=False))

---
## 5  ADWIN as a Learning Rate Controller

ADWIN is often used to trigger **model resets** in online classifiers.  
When drift is detected, the classifier re-initialises its hypothesis.

In [ ]:
from sklearn.linear_model import SGDClassifier
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score

def drifting_classification_stream(n=2000, drift_at=1000, seed=42):
    X0, y0 = make_classification(n_samples=drift_at, n_features=10, random_state=seed)
    X1, y1 = make_classification(n_samples=n - drift_at, n_features=10, random_state=seed+99)
    X1[:, :5] *= -1  # flip feature signs → concept drift
    return np.vstack([X0, X1]), np.hstack([y0, y1])

X, y = drifting_classification_stream()
CHUNK = 50

# Without ADWIN (pure incremental)
clf_no_adwin = SGDClassifier(loss='log_loss', random_state=42)
clf_no_adwin.fit(X[:CHUNK], y[:CHUNK])

# With ADWIN — reset on drift
clf_adwin = SGDClassifier(loss='log_loss', random_state=42)
clf_adwin.fit(X[:CHUNK], y[:CHUNK])
adwin_ctrl = ADWIN(delta=0.002)

accs_no, accs_adwin = [], []

for start in range(CHUNK, len(X) - CHUNK, CHUNK):
    Xc, yc = X[start:start+CHUNK], y[start:start+CHUNK]

    accs_no.append(accuracy_score(yc, clf_no_adwin.predict(Xc)))
    accs_adwin.append(accuracy_score(yc, clf_adwin.predict(Xc)))

    clf_no_adwin.partial_fit(Xc, yc, classes=[0, 1])

    # ADWIN monitors error rate
    err = 1 - accuracy_score(yc, clf_adwin.predict(Xc))
    if adwin_ctrl.add_element(err):
        clf_adwin = SGDClassifier(loss='log_loss', random_state=42)
        clf_adwin.fit(Xc, yc)  # reset
    else:
        clf_adwin.partial_fit(Xc, yc, classes=[0, 1])

plt.figure(figsize=(12, 4))
plt.plot(accs_no, label='Incremental (no reset)')
plt.plot(accs_adwin, label='ADWIN-controlled (reset on drift)', lw=2)
plt.axvline(1000 // CHUNK, color='black', ls='--', label='True drift')
plt.xlabel('Chunk index'); plt.ylabel('Accuracy')
plt.title('ADWIN as Model-Reset Controller')
plt.legend()
plt.tight_layout()
plt.show()

---
## Summary

| Property | Value |
|---|---|
| Memory | O(log n) with bucket compression |
| False alarm bound | Prob ≤ δ per step |
| Detection guarantee | Detects drift whenever true mean shift > ε |
| Main parameter | δ (confidence) — lower = more conservative |

**See also:** Topic 117 (stream adaptive learning), Topic 121 (drift detection methods: DDM, PHT)